# DA N-State Fit Template

This notebook is a template wrapper around the repository's DA ratio-fit workflow.
Edit the input block below, validate it, and then run the same backend used by the CLI.


## Imports / Setup

Run this notebook from the repository root, or adjust `REPO_ROOT` below.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parents[3]
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_da_fit_input_text,
    run_da_fit_from_notebook,
    validate_da_notebook_config,
)


## User Inputs

These fields mirror the plain-text DA input file format.
This template is structurally complete, but you should point it at your own HDF5 data and two-point n-state fit tables.


In [2]:
from pathlib import Path

EXAMPLE_C2PT = REPO_ROOT / "examples" / "l64c64a076_m140" / "data" / "c2pt_csv"
EXAMPLE_qDA = REPO_ROOT / "examples" / "l64c64a076_m140" / "data" / "qda"
EXAMPLE_2PT_RESULTS = REPO_ROOT / "examples" / "l64c64a076_m140" / "analysis" / "1-c2pt-fit" / "results_nstate_fit_2state"
EXAMPLE_OUTPUTS = REPO_ROOT / "examples" / "l64c64a076_m140" / "analysis" / "2-bm" / "results_fit_nst2"

workflow_config = {
    # Data settings
    'title_pattern': 'l64c64a076_m140_fit_pz*',
    'qda_h5': str(EXAMPLE_qDA / 'qDA_CG_1HYP_M140_GSRC_W45_k0_src5_O{gm}.h5'),
    'dataset_path_template': 'SP/{gm}/PX0PY0PZ{pz}/{Tdir}/{eta}/bT{bT}/bz{bz}',
    'tsrange': [0, 20],
    'ns': 64,
    'nt': 64,
    'lattice_spacing_fm': 0.076,
    'decay_constant_check': False,
    'pzlist': [0, 1],
    'gmlist': ['T5'],
    'etalist': ['eta0'],
    'Tdirlist': ['b_X'],
    'bTlist': [0],
    'bzlist': [bz for bz in range(0, 21)],

    # Two-point correlator input
    'two_point_fit_root': str(EXAMPLE_2PT_RESULTS),
    'two_point_fit_window_by_pz': {0: [3, 25], 1: [3, 25], 2: [3, 25]},
    'c2pt': str(EXAMPLE_C2PT / 'c2pt_5_5_k0_pz*_real.csv'),
    'fold_t': 'periodic',

    # DA fit settings
    'fit_target': 'ratio',
    'fit_component': 'both',
    'nstates': [2],
    'binsize': 10,
    'bootstrap_samples': 200,
    'bootstrap_size': 200,
    'seed': 2026,
    'two_point_fit_sample_coupled': True,
    'fit_window': {0: [4, 20], 1: [4, 20], 2: [4, 12]},
    'plot': True,
    'results_dir': str(EXAMPLE_OUTPUTS),
}

workflow_config


{'title_pattern': 'l64c64a076_m140_fit_pz*',
 'qda_h5': '/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/qda/qDA_CG_1HYP_M140_GSRC_W45_k0_src5_O{gm}.h5',
 'dataset_path_template': 'SP/{gm}/PX0PY0PZ{pz}/{Tdir}/{eta}/bT{bT}/bz{bz}',
 'tsrange': [0, 20],
 'ns': 64,
 'nt': 64,
 'lattice_spacing_fm': 0.076,
 'decay_constant_check': False,
 'pzlist': [0, 1],
 'gmlist': ['T5'],
 'etalist': ['eta0'],
 'Tdirlist': ['b_X'],
 'bTlist': [0],
 'bzlist': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20],
 'two_point_fit_root': '/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_2state',
 'two_point_fit_window_by_pz': {0: [3, 25], 1: [3, 25], 2: [3, 25]},
 'c2pt': '/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/c2pt_csv/c2pt_5_5_k0_pz*_real.csv',
 'fold_t': 'periodic',
 'fit_target': 'ratio',
 'fit_component': 

## Option Guide

Edit only `workflow_config` in the cell above for normal usage.
The keys are grouped by comments so data settings, fit settings, and output settings stay easy to scan.

- `title_pattern`: Output title pattern. Use `*` where the momentum index `pz` should be inserted.
- `ns`, `nt`: Spatial and temporal lattice extents.
- `lattice_spacing_fm`: Stored in metadata and summaries.
- `fit_target`: Keep this as `"ratio"` in the first implementation.
- `fit_component`: Choose `"real"`, `"imag"`, or `"both"`.
- `nstates`: Supported values are `1`, `2`, or `[1, 2]`.
- `pzlist`: Integer momentum labels to analyze.
- `gmlist`, `etalist`, `Tdirlist`: Lists passed into the HDF5 dataset-path expansion. Supported labels in the first version are `"T5"` for gamma_t gamma_5 and `"Z5"` for gamma_z gamma_5.
- `bTlist` / `bTrange`: Transverse-separation choices. Provide one style only.
- `bzlist` / `bzrange`: Longitudinal-separation choices. When `bz != 0`, the backend combines `+bz` and `-bz` automatically.
- `qda_h5`: HDF5 file path template. You may use `{pz}` and `{gm}` placeholders, or keep the legacy `*` pz wildcard. Example: `/path/to/qDA_..._O{gm}.h5` resolves to `..._OT5.h5` or `..._OZ5.h5` inside the main `(pz, gm)` loop.
- `dataset_path_template`: HDF5 dataset template with placeholders `{gm}`, `{eta}`, `{pz}`, `{Tdir}`, `{bT}`, and `{bz}`.
- `two_point_fit_root`: Root directory that contains the two-point n-state fit outputs. The backend looks under `<two_point_fit_root>/<title>/tables/` and selects the matching `_tmax<tmax>_fits.txt` file.
- `two_point_fit_window_by_pz`: Per-momentum dictionary that tells the backend which two-point `tmin` and `tmax` to reuse for each `pz`. In this notebook, write it as a Python dict such as `{0: [4, 12], 2: [5, 13]}`; the notebook helper materializes it into the tiny three-column mapping file expected by the parser.
- `c2pt`: Two-point correlator CSV used for the denominator of the ratio.
- `fold_t`: Folding mode for the denominator correlator. Use the same convention as the matching two-point analysis.
- Operator behavior: `T5` keeps the original sign-pattern preprocessing and numerator model. `Z5` multiplies the correlator by `-i` before folding and uses the extra lattice-momentum factor `Pz/E_i` in the numerator model.
- `tsrange`: Optional raw time range kept before fitting. If omitted, the backend defaults to `[0, Nt//2 - 1]`.
- `binsize`, `bootstrap_samples`, `bootstrap_size`, `seed`: Bootstrap controls.
- `fit_window`: The canonical window control. Use a dictionary like `{5: [6, 12], 6: [6, 12]}` to set one `[tmin, tmax]` window per momentum. A nested form like `{\"T5\": {5: [6, 12]}}` is also supported for `gm`-specific windows. The notebook helper materializes this into the backend fit-window table format automatically.
- Output grouping: for a fixed `(title, gm, eta, bT)`, the workflow writes one grouped ratio table containing all `bz` entries. For a fixed `(title, gm, eta, bT, component, nstates)`, it writes one grouped summary / fit / samples / curve file containing all `bz` entries.
- Grouped summaries contain one parseable `begin_bz ...` / `end_bz ...` block per `bz`, along with `two_point_fit_table_resolved`, `two_point_fit_tmax_source`, and `two_point_fit_tmax`. Those fields now reflect the explicit two-point fit-window config.
- Grouped fit tables include `two_point_fit_tmax` plus the usual fit columns.
- `plot`: Optional boolean. When `true`, also write grouped ratio-vs-fit PDF plots for each `(title, gm, eta, bT, component, nstates)` output.
- `results_dir`: Output directory. If set to `None`, notebook runs default to the notebook directory.


## Validate Config


In [3]:
parsed = validate_da_notebook_config(workflow_config)
parsed


DANStateInput(title_pattern='l64c64a076_m140_fit_pz*', ns=64, nt=64, lattice_spacing_fm=0.076, decay_constant_check=False, two_point_fit_sample_coupled=True, fit_target='ratio', fit_component='both', nstates=(2,), pzlist=(0, 1), gmlist=('T5',), etalist=('eta0',), tdirlist=('b_X',), bTlist=(0,), bzlist=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20), binsize=10, bootstrap_samples=200, bootstrap_size=200, seed=2026, fit_window='/var/folders/lp/9lcqf1_j5mldhw2q7rg992w00000gn/T/lqcd_da_fit_windows_fao7uvdz/da_fit_window.txt', qda_h5='/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/qda/qDA_CG_1HYP_M140_GSRC_W45_k0_src5_O{gm}.h5', dataset_path_template='SP/{gm}/PX0PY0PZ{pz}/{Tdir}/{eta}/bT{bT}/bz{bz}', c2pt='/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/c2pt_csv/c2pt_5_5_k0_pz*_real.csv', fold_t='periodic', tsrange=(0, 20), two_point_fit_root='/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/example

## Render Plain-Text Input Preview


In [4]:
print(render_da_fit_input_text(workflow_config))


l64c64a076_m140_fit_pz* 64 64 0.076
decay_constant_check false
fit_target ratio
fit_component both
nstates 2
pzlist 0 1
gmlist T5
etalist eta0
Tdirlist b_X
bTlist 0
bzlist 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20
fit_window /var/folders/lp/9lcqf1_j5mldhw2q7rg992w00000gn/T/lqcd_da_fit_windows_2n4qoe7c/da_fit_window.txt
qda_h5 /Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/qda/qDA_CG_1HYP_M140_GSRC_W45_k0_src5_O{gm}.h5
dataset_path_template SP/{gm}/PX0PY0PZ{pz}/{Tdir}/{eta}/bT{bT}/bz{bz}
two_point_fit_root /Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_2state
two_point_fit_window_by_pz /var/folders/lp/9lcqf1_j5mldhw2q7rg992w00000gn/T/lqcd_da_two_point_fit_window_84lkav1y/two_point_fit_window_by_pz.txt
c2pt /Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/c2pt_csv/c2pt_5_5_k0_pz*_real.csv
fold_t periodic
tsrange 0 20
binsize 10
bootstrap_samples 20

## Run Backend Workflow


In [5]:
outputs = run_da_fit_from_notebook(workflow_config)
for output in outputs:
    print(output)


/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/2-bm/results_fit_nst2/l64c64a076_m140_fit_pz0/tables/l64c64a076_m140_fit_pz0_T5_eta0_bT0_ratio.txt
/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/2-bm/results_fit_nst2/l64c64a076_m140_fit_pz0/l64c64a076_m140_fit_pz0_T5_eta0_bT0_real_2state_summary.txt
/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/2-bm/results_fit_nst2/l64c64a076_m140_fit_pz0/tables/l64c64a076_m140_fit_pz0_T5_eta0_bT0_real_2state_fit.txt
/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/2-bm/results_fit_nst2/l64c64a076_m140_fit_pz0/samples/l64c64a076_m140_fit_pz0_T5_eta0_bT0_real_2state_samples.txt
/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/2-bm/results_fit_nst2/l64c64a076_m140_fit_pz0/tables/l64c64a076_m140_fit_pz0_T5_eta0_bT0_real_2state_curve.txt
/Users/xiang/Desktop/codes/lat-hadron-ana

## Config Snapshot


In [6]:
print(pretty_print_config(workflow_config))


{
  "title_pattern": "l64c64a076_m140_fit_pz*",
  "qda_h5": "/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/qda/qDA_CG_1HYP_M140_GSRC_W45_k0_src5_O{gm}.h5",
  "dataset_path_template": "SP/{gm}/PX0PY0PZ{pz}/{Tdir}/{eta}/bT{bT}/bz{bz}",
  "tsrange": [
    0,
    20
  ],
  "ns": 64,
  "nt": 64,
  "lattice_spacing_fm": 0.076,
  "decay_constant_check": false,
  "pzlist": [
    0,
    1
  ],
  "gmlist": [
    "T5"
  ],
  "etalist": [
    "eta0"
  ],
  "Tdirlist": [
    "b_X"
  ],
  "bTlist": [
    0
  ],
  "bzlist": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20
  ],
  "two_point_fit_root": "/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/1-c2pt-fit/results_nstate_fit_2state",
  "two_point_fit_window_by_pz": {
    "0": [
      3,
      25
    ],
    "1": [
      3,
      25
    ],
    "2": [
      3,
      25
    